In [1]:

import sys
import os
os.environ['HF_HOME'] = '/projectnb/vkolagrp/skowshik/.cache/'
import logging
import argparse
import warnings
warnings.filterwarnings("ignore")
import yaml
import torch
import json
import inspect
import pandas as pd
import torch.distributed as dist
import torchtune
import re

from tqdm import tqdm
from torch.nn.parallel import DistributedDataParallel as DDP
from datetime import datetime, timedelta
from huggingface_hub import login
from lib.model_loader import load_model
from lib.trainer_loader import load_trainer
from lib.data_loader import data_loader_
from utils.utils import CustomStream, load_config
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, AutoConfig, AutoModel
from lib.model_loader_eval import load_model_eval, save_merged_model, load_model_with_llama_proj
from utils.utils import load_json, load_csv
# from vllm import LLM, SamplingParams
# from vllm.lora.request import LoRARequest
from safetensors.torch import load_file
from tokenizers import AddedToken
# from peft import LoraConfig, TaskType, PeftModel, get_peft_model
from collections import defaultdict
# from lib.modeling_llama import LlamaVisionForCausalLM, LlamaForCausalLM

In [2]:
# config = load_config(file_name="../../code/training/config/config_imaging.yml")
config = load_config(file_name="../../code/training/config/config_small.yml")
sysmsg = config.get("sysmsg")
dataset_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/0825/llama_summaries/nacc_unique_with_llama_summaries_test.json"
data = load_json(dataset_path)
# data_train = load_json("../../data/0825/nacc_unique_with_llama_summaries_train.json")
# data = load_csv(dataset_path)
max_new_tokens = config.get("max_new_tokens")
hf_write_token = config.get("hf_write_token")
n_devices = 1
max_new_tokens = 4096

In [ ]:
nacc_np = pd.read_csv('/projectnb/vkolagrp/datasets/NACC/csv/raw/investigator_ftldlbd_merged_neuropath_nacc65.csv')
# nacc = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/new_nacc_unique_type_3.csv")

In [ ]:
nacc_np[nacc_np['NACCID'] == 'NACC181584'][['NACCUDSD', 'NACCETPR', 'NACCALZP', 'NPTHAL', 'NACCBRAA', 'NPCERAD']]

,NACCUDSD,NACCETPR,NACCALZP,NPTHAL,NACCBRAA,NPCERAD
20524,1,88,8,3,5,NaN
20525,1,88,8,3,5,NaN
20526,3,1,1,3,5,NaN
20527,3,1,1,3,5,NaN
20528,3,1,1,3,5,NaN
20529,4,1,1,3,5,NaN
20530,4,1,1,3,5,NaN
20531,4,1,1,3,5,NaN


In [3]:
example_NACC181584 = load_json("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/0825/csv_to_txt/example_NACC181584.json")

In [4]:
def get_mri_or_emb_dict(val):
    mris_found = defaultdict(int)
    if val == 'mri':
        data_path = '/projectnb/vkolagrp/datasets/NACC/MRI/skullstripped'
        save_path = 'nacc_mri_paths.json'
        extension = '.nii'
    else:
        data_path = '/projectnb/vkolagrp/datasets/NACC/MRI/swin_emb'
        save_path = 'nacc_emb_paths.json'
        extension = '.npy'
    j = 6
    def extract_id(name):
        # This regex matches any characters following "fname-" up until the first occurrence of "_mo"
        match = re.search(r'fname-(.*?)_mo', name)
        if match:
            return match.group(1)    # Returns the captured group which is the part of the string you need
        else:
            match = re.search(r'fname-(.*?)_nodate', name)
            if match:
                return match.group(1) 
        return None

    mri_dict = {}
    # for dirpath, dirs, files in tqdm(os.walk('/projectnb/vkolagrp/datasets/NACC/MRI/skullstripped')): 
    for dirpath, dirs, files in tqdm(os.walk(data_path)): 
        if len(files) == 0:
            continue
        for filename in files:
            fname = os.path.join(dirpath, filename)
            path_list = fname.split('/')
            naccid = path_list[j+1]
            # print(path_list)
            zip_name = extract_id(path_list[j+2])
            modality = path_list[j+3]
            acquisition = path_list[j+4]
            mri_name = path_list[j+5]
            if mri_name in mris_found:
                continue
            mris_found[mri_name] += 1
            # print(naccid, zip_name, modality, acquisition, mri_name)
            
            if modality == 'Unknown' or (not mri_name.endswith(extension)):
                continue
            # if naccid not in mri_dict:
            #     mri_dict[naccid] = {}
            
            if zip_name + 'ni' not in mri_dict:
                mri_dict[zip_name + 'ni'] = {}
            
            mri_dict[zip_name + 'ni'][modality] = {"acquisition": acquisition, "mri_name": fname}


    # with open(save_path, "w") as outfile:
    #     json.dump(mri_dict, outfile, indent=4, sort_keys=False)
        
    return mri_dict, mris_found

In [5]:
emb_dict, mris_found = get_mri_or_emb_dict(val='emb')

21263it [00:11, 1905.51it/s]


In [6]:
import copy
modified_data = copy.deepcopy(example_NACC181584)
for i, entry in enumerate(modified_data):
    zip_name = entry['NACCMRFI']
    if isinstance(zip_name, str):
        mod_zip_name = zip_name.replace(".zip", "ni")
        print(mod_zip_name)
        if mod_zip_name in emb_dict:
            
            print("Modifying")
            mri_text = ''
            for seq, info in emb_dict[mod_zip_name].items():
                if "diffusion" in seq.lower():
                    continue
                mri_text += f"{seq} MRI image: {info['mri_name']} "
            
            # print(mri_text)
            modified_data[i]['patient_LLAMA_SUMMARY'] += f"\n\n**MRI imaging**:\n\n{mri_text}"
            # print(modified_data[i]['patient_LLAMA_SUMMARY'])
            # break

NACC181584_1284011361925176253428328511104770718814ni
NACC181584_128401136192134176253428319551235484334303ni
NACC181584_128401136192134176253428334941297274221731ni
NACC181584_128401136192134176253428338181342538475427ni
Modifying


In [7]:
# for entry in modified_data:
#     if 'project' in entry['patient_LLAMA_SUMMARY']:
#         print(entry['patient_LLAMA_SUMMARY'])

## Load model

In [8]:
# llm, tokenizer = load_model_eval_vllm(config, n_devices=2, enable_lora=False, torch_dtype="bfloat16", new_model_path=config.get("model_name"))

In [9]:
# model, tokenizer = load_model_eval(config, base_model=config.get("model_name"), lora_path=config.get("hub_model_id"), torch_dtype=torch.bfloat16, vision=True)
model, tokenizer = load_model_eval(config, base_model=config.get("hub_final_model_id"), lora_path=None, torch_dtype=torch.bfloat16, vision=True)#, save_dir='../../ckpt/fine_tuned_v6_end_to_end_with_mri')

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


Loading LlamaVisionForCausalLM


You are using a model of type llamavision to instantiate a model of type llama. This is not supported for all configurations of models and can yield errors.
Loading checkpoint shards: 100%|██████████| 4/4 [00:38<00:00,  9.56s/it]


Copying SwinUNeTR model weights


In [9]:
torch.cuda.current_device()

0

In [10]:
import re
import numpy as np

def add_image_tokens(df):
    pattern = r'(/\S+\.npy)'
    all_matches = []

    # Iterate over each row in the DataFrame
    for entry in data:
        # Find all matches of the pattern in the 'user' column
        matches = re.findall(pattern, entry['patient_LLAMA_SUMMARY'])
        all_matches += matches
    
    all_matches = list(set(all_matches))
    for match in all_matches:
        try:
            x = np.load(match, mmap_mode='r')
        except:
            print(match)
            raise ValueError
    print(len(all_matches))
    return all_matches

all_matches = add_image_tokens(data)
# all_matches.append('/projectnb/vkolagrp/datasets/NACC/MRI/skullstripped/NACC004099/fname-NACC004099_128401136192134176253428319421300140307886_mo-4_dy-27_yr-2011/T1/3D/1.2.840.113619.2.134.1762534283.1942.1300140307.908_stripped.nii')

new_tokens = set()
if all_matches:
    for matched_string in all_matches:
        token = AddedToken(matched_string, lstrip=False, rstrip=False, normalized=False, special=False)
        new_tokens.add(token)
    tokenizer.add_tokens(list(new_tokens))
    model.resize_token_embeddings(len(tokenizer))

942


In [11]:
def process_user_message(user_msg, reserved_tokens, tokenizer):
    # Define the regular expression pattern for words ending with .nii
    pattern = r'(/\S+\.npy)'
    
    # Define the replacement function
    # It adds "start_of image" before the word, quotes around the word, and "{reserved_tokens}end_of_mri" after
    def replace_func(match):
        matched_string = match.group(1)
        
        # Print the matched string
        # print("Matched string:", matched_string)
        return f'<|start_of_mri|>{matched_string}{reserved_tokens}<|end_of_mri|>'
    
    # Use re.sub to replace all occurrences in the input message
    processed_msg = re.sub(pattern, replace_func, user_msg)
    # print(processed_msg)
    
    return processed_msg

## Testing

In [12]:
index = 5
print(modified_data[index]['patient_LLAMA_SUMMARY'])

**Medical History Summary**

**Subject Demographics:**

* Age: 81 years
* Sex: Female
* Primary language: Spanish
* Hispanic/Latino ethnicity: Yes
* Marital status: Married
* Years of education: 14
* Living situation: Lives with spouse or partner

**Subject Family History:**

* Father: No report of father with cognitive impairment
* Mother: No report of mother with cognitive impairment
* First-degree family member: Report of at least one first-degree family member with cognitive impairment

**Subject Medications:**

* Total number of medications reported at each visit: 2
* Current use of an angiotensin converting enzyme (ACE) inhibitor: Yes
* Current use of any type of antihypertensive or blood pressure medication: Yes
* Current use of lipid lowering medication: Yes
* Subject taking any medications: Yes

**Subject Health History:**

* Brain trauma - chronic deficit: Absent
* Depression episodes more than two years ago: No
* Brain trauma - brief unconsciousness: Absent
* Brain trauma - 

### Question: Provide the cognitive status at UDS visit for a patient presenting with the following information.

In [13]:
index = 1
print(modified_data[index]['NACCID'])
print(modified_data[index]['NACCUDSD'])

reserved_tokens = '<|reserved_mri_token|>' * (config.get('k') - 1)
user_msg = process_user_message(modified_data[index]['patient_LLAMA_SUMMARY'], reserved_tokens, tokenizer)

NACC181584
1


In [14]:
# messages = [
#     [
#         {"role": "system", "content": sysmsg},
#         {"role": "user", "content": f"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nProvide the cognitive status at UDS visit for a patient presenting with the following information.\n\n### Input:\n{user_msg}"}
#     ],
# ]

messages = [
    [
        {"role": "system", "content": sysmsg},
        {"role": "user", "content": f"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nDoes this patient have normal cognition? Just answer Yes or No. \n\n### Input:\n{user_msg}"}
    ],
]

In [20]:
prompt = tokenizer.apply_chat_template(
    messages[0], 
    # tokenize=False, 
    return_tensors="pt",
    add_generation_prompt=True
)#.to("cuda")

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = model.generate(
        prompt,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.1,
    )
response = outputs[0][prompt.shape[-1]:]
print(tokenizer.decode(response, skip_special_tokens=True).replace(" (NINDS/AIREN criteria)", "").replace(" UDS", ""))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


Yes.


In [19]:
prompt

tensor([[128000, 128006,   9125,  ...,  78191, 128007,    271]])

### Question: Provide the primary etiologic diagnosis for a patient presenting with the following information.

In [23]:
index = -1
print(modified_data[index]['NACCID'])
print(modified_data[index]['NACCUDSD'])
print(modified_data[index]['NACCETPR'])

reserved_tokens = '<|reserved_mri_token|>' * (config.get('k') - 1)
user_msg = process_user_message(modified_data[index]['patient_LLAMA_SUMMARY'], reserved_tokens, tokenizer)

NACC181584
4
1


In [24]:
messages = [
    [
        {"role": "system", "content": sysmsg},
        {"role": "user", "content": f"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nProvide the primary etiologic diagnosis for a patient presenting with the following information.\n\n### Input:\n{user_msg}"}
    ],
]

In [25]:
prompt = tokenizer.apply_chat_template(
    messages[0], 
    # tokenize=False, 
    return_tensors="pt",
    add_generation_prompt=True
).to("cuda")

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = model.generate(
        prompt,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.2,
    )
response = outputs[0][prompt.shape[-1]:]
print(tokenizer.decode(response, skip_special_tokens=True))

Based on the provided information, the primary etiologic diagnosis for this patient is Alzheimer's disease (AD). The patient's cognitive status is deemed to be dementia, and the presumptive etiologic diagnosis is Alzheimer's disease. The patient's symptoms, including memory loss, language difficulties, and visuospatial impairment, are consistent with AD. Additionally, the patient's age, family history, and genetic testing (APOE e4 allele) do not suggest any other potential causes of cognitive impairment. The patient's medications, including lisinopril and darifenacin, are not likely to be contributing to their cognitive impairment. Overall, the evidence suggests that Alzheimer's disease is the most likely cause of the patient's cognitive impairment.


Based on the provided information, the primary etiologic diagnosis for this patient is Alzheimer's disease (AD). The patient's cognitive status is deemed to be dementia, and the presumptive etiologic diagnosis is Alzheimer's disease. The patient's symptoms, including memory loss, language difficulties, and visuospatial impairment, are consistent with AD. Additionally, the patient's age, family history, and genetic testing (APOE e4 allele) do not suggest any other potential causes of cognitive impairment. The patient's medications, including lisinopril and darifenacin, are not likely to be contributing to their cognitive impairment. Overall, the evidence suggests that Alzheimer's disease is the most likely cause of the patient's cognitive impairment.

### Question: Is my patient at risk of progression to dementia?

In [26]:
index = 2
print(modified_data[index]['NACCID'])

reserved_tokens = '<|reserved_mri_token|>' * (config.get('k') - 1)
user_msg = process_user_message(modified_data[index]['patient_LLAMA_SUMMARY'], reserved_tokens, tokenizer)

NACC181584


In [27]:
messages = [
    [
        {"role": "system", "content": sysmsg},
        {"role": "user", "content": f"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nWhat is the cognitive diagnosis for a patient presenting with this information. Are they at risk of progressing to dementia in the next ten years? The MRI images are provided between tokens <|start_of_mri|> and <|end_of_mri|>.\n\n### Input:\n{user_msg}"}
    ],
]

In [28]:
prompt = tokenizer.apply_chat_template(
    messages[0], 
    # tokenize=False, 
    return_tensors="pt",
    add_generation_prompt=True
).to("cuda")

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = model.generate(
        prompt,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.2,
    )
response = outputs[0][prompt.shape[-1]:]
print(tokenizer.decode(response, skip_special_tokens=True))

Based on the provided information, the cognitive diagnosis for this patient is Mild Cognitive Impairment (MCI) due to Alzheimer's disease (AD). The patient is a 77-year-old female with a history of hypertension, hypercholesterolemia, and osteoporosis, but no history of stroke, traumatic brain injury, or other neurological conditions. She has a family history of cognitive impairment, but no first-degree relatives with dementia.

The patient's cognitive status is deemed to be one or two test scores abnormal, with a total MMSE score of 27 out of 30. She has difficulty with memory, particularly with recalling recent events and learning new information. She also has difficulty with language, particularly with word-finding and naming. Her executive function is normal, and she has no difficulty with attention or concentration.

The patient's imaging evidence is normal, and she has no evidence of Lewy body disease, frontotemporal dementia, or other neurodegenerative disorders. Her genetic test

### Question: Provide the contributing factors for the cognitive decline for a patient presenting with the following information.

In [ ]:
index = 10228
print(data[index]['NACCID'])
print(data[index]['NACCUDSD'])
print(data[index]['instruction'])

reserved_tokens = '<|reserved_mri_token|>' * (config.get('k') - 1)
user_msg = process_user_message(data[index]['patient_LLAMA_SUMMARY'], reserved_tokens, tokenizer)

In [ ]:
messages = [
    [
        {"role": "system", "content": sysmsg},
        {"role": "user", "content": f"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\n{data[index]['instruction']} The MRI images are provided between tokens <|start_of_mri|> and <|end_of_mri|>.\n\n### Input:\n{user_msg}"}
    ],
]

In [ ]:
prompt = tokenizer.apply_chat_template(
    messages[0], 
    # tokenize=False, 
    return_tensors="pt",
    add_generation_prompt=True
).to("cuda")

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = model.generate(
        prompt,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.2,
    )
response = outputs[0][prompt.shape[-1]:]
print(tokenizer.decode(response, skip_special_tokens=True))